# Libs

In [35]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from astropy.visualization import ZScaleInterval
from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
from astropy.utils.exceptions import AstropyWarning
from pathlib import Path
from typing import Tuple, Dict, Optional, Union
from matplotlib.lines import Line2D
import matplotlib.patches as patches
from scipy.ndimage import distance_transform_edt
import cv2
from astropy.stats import sigma_clipped_stats
from scipy.ndimage import distance_transform_edt
from astropy.visualization import ZScaleInterval, AsinhStretch, ImageNormalize
import json
import pandas as pd
from __future__ import annotations
    

# Plot params

In [36]:
# ====================================================================
# Publication-quality plot settings (ApJ, MNRAS, A&A standards)
# ====================================================================
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "Times New Roman"],
    "axes.labelsize": 14,
    "axes.titlesize": 14,
    "legend.fontsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "axes.linewidth": 1.2,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight"
})

# Objects params

In [37]:
# Ваш словарь объектов
obj_dist_step_pc = {
    "Auriga": [450.0, 3, "Lada et al. 2009"],
    "L1495": [140.0, 3, "Galli et al. 2018 / Schlafly et al. 2014"],
    "L43_Karoly_2023": [125.0, 2, "Ortiz-León et al. 2017"],
    "MonR2": [830.0, 3, "Dzib et al. 2016"],
    "NGC1333": [299.0, 3, "Zucker et al. 2018"],
    "OMC1_450": [388.0, 3, "Kounkel et al. 2017"],
    "OMC1": [388.0, 3, "Kounkel et al. 2017"], # ± 5 pc
    "Perseus_B1": [293.0, 3, "Zucker et al. 2018"],
    "Rosette": [1330.0, 3, "Heyer et al. 2006"],
    "l1689b": [147.0, 3, "Ortiz-León et al. 2017"],
    "oph_a": [139.0, 3, "Zucker et al. 2019"],
    "oph_b": [139.0, 3, "Zucker et al. 2019"],
    "oph_c": [139.0, 3, "Zucker et al. 2019"]
}

# Funcs

In [41]:
from email import header


class PolarizationAnalyzer:
    """
    Pipeline for analyzing dust polarization maps. 
    Focuses on generating structural maps and the polarization angle 
    structure function without assuming an underlying turbulence model.
    """
    def __init__(self, base_archive_path: Union[str, Path]):
        """
        Initializes the analyzer with the root directory of the archive.
        
        Args:
            base_archive_path (str or Path): Path to the BISTRO data archive.
            
        Raises:
            FileNotFoundError: If the provided directory does not exist.
        """
        self.base_dir = Path(base_archive_path)
        if not self.base_dir.exists():
            raise FileNotFoundError(f"Base directory not accessible: {self.base_dir}")
        
    def _load_fits(self, file_path: Path) -> Tuple[np.ndarray, Optional[WCS]]:
        """
        Loads FITS data and collapses degenerate axes.
        
        Args:
            file_path (Path): Path to the target FITS file.
            
        Returns:
            Tuple[np.ndarray, Optional[WCS]]: 2D image array and its celestial WCS.
            
        Raises:
            FileNotFoundError: If the FITS file is missing.
        """
        if not file_path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")
            
        with fits.open(file_path) as hdul:
            data = np.squeeze(hdul[0].data)
            header = hdul[0].header
            
            with warnings.catch_warnings():
                warnings.simplefilter('ignore', AstropyWarning)
                wcs = WCS(header) if 'CTYPE1' in header else None
                if wcs is not None and wcs.pixel_n_dim > 2:
                    wcs = wcs.celestial
        return data, wcs
    
    def _get_robust_limits(self, I: np.ndarray) -> Tuple[float, float]:
        """
        Safely computes display limits, falling back to percentiles if ZScale fails.
        """
        valid_I = I[np.isfinite(I)]
        if valid_I.size == 0:
            return 0.0, 1.0 # Fallback for completely empty maps
            
        try:
            interval = ZScaleInterval()
            vmin, vmax = interval.get_limits(valid_I)
            # Ensure vmax is strictly greater than vmin
            if vmax <= vmin:
                raise ValueError("ZScale produced invalid limits.")
        except Exception:
            # Fallback to robust percentiles
            vmin, vmax = np.percentile(valid_I, [2, 98])
            
        # Absolute final safety check
        if vmax <= vmin:
            vmin = np.nanmin(valid_I)
            vmax = np.nanmax(valid_I)
            if vmax <= vmin: # Map is perfectly flat
                 vmax = vmin + 1.0 
                 
        return vmin, vmax
    
    def load_stokes_maps(self, object_name: str, band: str = "850um") -> Tuple[np.ndarray, np.ndarray, np.ndarray, WCS]:
        """
        Loads Stokes I, Q, and U maps for a specific target.
        
        The method first attempts to find files using the strict BISTRO naming convention:
        '{object_name}_{band}_{stokes}_map.fits'. 
        If strict files are not found, it falls back to a flexible search for any 
        FITS file containing '{stokes}_map' in its filename, accommodating both 
        '.fits' and '.fit' extensions.

        Args:
            object_name (str): Target directory name.
            band (str): Observation band (e.g., "850um").
            
        Returns:
            Tuple[np.ndarray, np.ndarray, np.ndarray, WCS]: Synchronized I, Q, U maps and WCS.
            
        Raises:
            FileNotFoundError: If any of the Stokes components cannot be located.
        """
        maps_dir = self.base_dir / object_name / "maps"
        if not maps_dir.exists():
            raise FileNotFoundError(f"Maps directory not found: {maps_dir}")

        stokes_data = {}
        for component in ['I', 'Q', 'U']:
            # 1. Primary strict search
            strict_name = f"{object_name}_{band}_{component}_map.fits"
            file_path = maps_dir / strict_name
            
            # 2. Fallback flexible search (now accepting .fit and .fits)
            if not file_path.exists():
                flexible_pattern = f"*{component}_map*.fit*"
                found_files = list(maps_dir.glob(flexible_pattern))
                
                if found_files:
                    file_path = found_files[0]  # Take the first matching file
                else:
                    raise FileNotFoundError(
                        f"Could not find Stokes {component} map for {object_name}. "
                        f"Tried strict '{strict_name}' and pattern '{flexible_pattern}'."
                    )
            
            data, wcs_loaded = self._load_fits(file_path)
            stokes_data[component] = data
            
            # Use WCS from the Stokes I map as the reference
            if component == 'I':
                wcs = wcs_loaded

        I, Q, U = stokes_data['I'], stokes_data['Q'], stokes_data['U']

        # Shape synchronization check to prevent broadcasting errors
        if Q.shape != I.shape or U.shape != I.shape:
            print(f"[Warning] Shape mismatch for {object_name}. Synchronizing Q/U to I: {I.shape}")
            ny, nx = I.shape
            Q = Q[:ny, :nx]
            U = U[:ny, :nx]
            
        return I, Q, U, wcs
    
    def load_hawc_stokes_maps(self, fits_path):
        with fits.open(fits_path) as hdul:
            I  = np.squeeze(hdul["STOKES I"].data).astype(float)
            dI = np.squeeze(hdul["ERROR I"].data).astype(float)
    
            Q  = np.squeeze(hdul["STOKES Q"].data).astype(float)
            dQ = np.squeeze(hdul["ERROR Q"].data).astype(float)
    
            U  = np.squeeze(hdul["STOKES U"].data).astype(float)
            dU = np.squeeze(hdul["ERROR U"].data).astype(float)
    
            badpix = np.squeeze(hdul["BAD PIXEL MASK"].data)
    
            wcs = WCS(hdul["STOKES I"].header).celestial
    
            obj = hdul["STOKES I"].header.get("OBJECT", Path(fits_path).stem).strip()
    
        var_I = dI**2
        var_Q = dQ**2
        var_U = dU**2
    
        qual_I = badpix
        qual_Q = badpix
        qual_U = badpix
    
        return obj, I, Q, U, wcs, var_I, var_Q, var_U, qual_I, qual_Q, qual_U
    
    def load_fireplace_cutout(self, cutout_path):
        from pathlib import Path
        import numpy as np
        from astropy.io import fits
        from astropy.wcs import WCS

        cutout_path = Path(cutout_path)

        with fits.open(cutout_path) as hdul:
            names = [h.name.strip().upper() for h in hdul]

            def get_hdu_data(possible_names, required=True):
                for name in possible_names:
                    name_up = name.upper()
                    if name_up in names:
                        return np.squeeze(hdul[name_up].data).astype(float)
                if required:
                    raise KeyError(f"Could not find any of these HDUs: {possible_names}")
                return None

            I = get_hdu_data(["STOKES I", "I"])
            Q = get_hdu_data(["STOKES Q", "Q"])
            U = get_hdu_data(["STOKES U", "U"])

            dI = get_hdu_data(["ERROR I"], required=False)
            dQ = get_hdu_data(["ERROR Q"], required=False)
            dU = get_hdu_data(["ERROR U"], required=False)

            image_mask = get_hdu_data(["IMAGE MASK", "MASK", "BAD PIXEL MASK", "QUALITY"], required=False)

            pol_flux = get_hdu_data(["POL FLUX"], required=False)
            err_pol_flux = get_hdu_data(["ERROR POL FLUX"], required=False)
            deb_pol_flux = get_hdu_data(["DEBIASED POL FLUX"], required=False)

            pol_angle = get_hdu_data(["POL ANGLE"], required=False)
            rot_pol_angle = get_hdu_data(["ROTATED POL ANGLE"], required=False)
            err_pol_angle = get_hdu_data(["ERROR POL ANGLE"], required=False)

            if "STOKES I" in hdul:
                header = hdul["STOKES I"].header
            else:
                header = hdul[0].header

            wcs = WCS(header).celestial
            
            # FOR FIREPLACE CUTOUTS, the object name is not reliably stored in the header, so we use the filename as a fallback.
            # regname = header.get("REGNAME", cutout_path.parent.name)
            # object_name = str(regname).strip().replace(" ", "_")

            # FOR FIELDMAPS, the object name is stored in the header under "REGNAME" or "OBJECT", so we check those first.
            regname = header.get("REGNAME")
            objname = header.get("OBJECT")

            name = regname if regname not in [None, ""] else objname
            if name in [None, ""]:
                name = cutout_path.stem

            object_name = str(name).strip().replace(" ", "_")

        data = {
            "object_name": object_name,
            "I": I,
            "Q": Q,
            "U": U,
            "dI": dI,
            "dQ": dQ,
            "dU": dU,
            "mask": image_mask,
            "pol_flux": pol_flux,
            "err_pol_flux": err_pol_flux,
            "deb_pol_flux": deb_pol_flux,
            "pol_angle": pol_angle,
            "rot_pol_angle": rot_pol_angle,
            "err_pol_angle": err_pol_angle,
            "wcs": wcs,
        }

        return data

    @staticmethod
    def compute_evpa(Q: np.ndarray, U: np.ndarray) -> np.ndarray:
        """
        Computes the Electric Vector Position Angle (EVPA).
        
        Args:
            Q (np.ndarray): Stokes Q map.
            U (np.ndarray): Stokes U map.
            
        Returns:
            np.ndarray: EVPA map in degrees (-90 to +90).
        """
        return np.degrees(0.5 * np.arctan2(U, Q))

    @staticmethod
    def pol_vector_components(p: Union[float, np.ndarray], psi_rad: Union[float, np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:
        """
        Calculates Cartesian vector components for plotting the B-field.
        Includes a +pi/2 rotation to convert EVPA to magnetic field orientation.
        
        Args:
            p (float or np.ndarray): Polarization degree or constant length.
            psi_rad (float or np.ndarray): EVPA in radians.
            
        Returns:
            Tuple[np.ndarray, np.ndarray]: X and Y vector components (vx, vy).
        """
        psi_shifted = psi_rad + np.pi / 2.0
        return p * np.cos(psi_shifted), p * np.sin(psi_shifted)

    @staticmethod
    def estimate_rms(data: np.ndarray) -> float:
        """
        Robustly estimates the Root Mean Square (RMS) noise using the Median Absolute Deviation (MAD).
        
        Args:
            data (np.ndarray): 2D map array (e.g., Stokes I or polarized intensity).
            
        Returns:
            float: Estimated background noise level. Returns 0.0 if map is empty/invalid.
        """
        valid = data[np.isfinite(data) & (data != 0)]
        if valid.size == 0: return 0.0
        return 1.4826 * np.median(np.abs(valid - np.median(valid)))

    def get_physical_scales(self, wcs: WCS, distance_pc: float, shape: Tuple[int, int]) -> Dict[str, float]:
        """
        Converts pixel grid scales to physical dimensions.
        
        Args:
            wcs (WCS): World Coordinate System object.
            distance_pc (float): Distance to the source in parsecs.
            shape (Tuple[int, int]): Dimensions of the image (Ny, Nx).
            
        Returns:
            Dict[str, float]: Dictionary containing pixel scales (arcsec, pc) and total field of view.
        """
        pix_scale_deg = np.mean(proj_plane_pixel_scales(wcs))
        pix_scale_arcsec = pix_scale_deg * 3600.0
        pix_scale_pc = np.radians(pix_scale_deg) * distance_pc
        
        return {
            "distance_pc": distance_pc,
            "pix_scale_arcsec": pix_scale_arcsec,
            "phys_scale_pc": pix_scale_pc,
            "angular_size_arcmin": (shape[0] * pix_scale_deg * 60, shape[1] * pix_scale_deg * 60),
            "physical_size_pc": (shape[0] * pix_scale_pc, shape[1] * pix_scale_pc),
            "total_pixels": shape[0] * shape[1]
        }

    @staticmethod
    def compute_structure_function(Q: np.ndarray, U: np.ndarray, R_list: np.ndarray, 
                                   min_pI: Optional[float] = None, bin_width: float = 0.5) -> Dict[str, np.ndarray]:
        """
        Computes the polarization angle structure function D_phi(R) = <sin^2(phi_1 - phi_2)>.
        
        Args:
            Q (np.ndarray): Stokes Q map.
            U (np.ndarray): Stokes U map.
            R_list (np.ndarray): Array of pixel lags (radii) to compute the function over.
            min_pI (Optional[float]): Minimum polarized intensity threshold. Defaults to None.
            bin_width (float): Tolerance window for pixel pair distance matching. Defaults to 0.5.
            
        Returns:
            Dict[str, np.ndarray]: Dictionary with keys 'R' (radii), 'Dphi' (structure function), 'Npairs' (counts).
        """
        Q, U = np.asarray(Q, dtype=float), np.asarray(U, dtype=float)
        Ny, Nx = Q.shape
        pI = np.hypot(Q, U)
        
        if min_pI is None:
            min_pI = -np.inf
            
        valid = np.isfinite(Q) & np.isfinite(U) & (pI > min_pI)
        q = np.where(valid, Q / pI, np.nan)
        u = np.where(valid, U / pI, np.nan)

        Dphi = np.full_like(R_list, np.nan, dtype=float)
        Npairs = np.zeros_like(R_list, dtype=int)

        def accumulate_shift(dy: int, dx: int) -> Optional[np.ndarray]:
            y1s, y2s = slice(max(0, dy), min(Ny, Ny + dy)), slice(max(0, -dy), min(Ny, Ny - dy))
            x1s, x2s = slice(max(0, dx), min(Nx, Nx + dx)), slice(max(0, -dx), min(Nx, Nx - dx))
            q1, u1 = q[y1s, x1s], u[y1s, x1s]
            q2, u2 = q[y2s, x2s], u[y2s, x2s]
            m = np.isfinite(q1) & np.isfinite(q2)
            if not np.any(m): return None
            dq, du = q1[m] - q2[m], u1[m] - u2[m]
            return 0.25 * (dq**2 + du**2)

        for i, R in enumerate(R_list):
            if R < 0: continue
            rmax = int(np.ceil(R + bin_width))
            shifts = [(dy, dx) for dy in range(-rmax, rmax + 1) for dx in range(-rmax, rmax + 1) 
                      if (dx != 0 or dy != 0) and abs(np.hypot(dx, dy) - R) <= bin_width]

            contrib_list = [accumulate_shift(dy, dx) for dy, dx in shifts]
            valid_contribs = [c for c in contrib_list if c is not None and c.size > 0]
            if valid_contribs:
                contrib_vec = np.concatenate(valid_contribs)
                Dphi[i] = np.nanmean(contrib_vec)
                Npairs[i] = contrib_vec.size

        return {"R": R_list, "Dphi": Dphi, "Npairs": Npairs}
    
    def build_fireplace_masks(
        self,
        I,
        Q,
        U,
        var_Q=None,
        var_U=None,
        mask=None,
        snr_main=2.0,
        snr_strict=3.0,
        bright_percentile=75,
    ):
        import numpy as np

        finite_mask = (
            np.isfinite(I)
            & np.isfinite(Q)
            & np.isfinite(U)
        )

        if mask is not None:
            # assumes 0 = good, non-zero = bad
            good_mask = finite_mask & (mask == 0)
        else:
            good_mask = finite_mask

        pI_raw = np.hypot(Q, U)

        if var_Q is not None and var_U is not None:
            err_Q = np.sqrt(np.where(var_Q > 0, var_Q, np.inf))
            err_U = np.sqrt(np.where(var_U > 0, var_U, np.inf))

            pI_safe = np.where(pI_raw > 0, pI_raw, np.inf)

            noise_pI = np.sqrt(
                Q**2 * err_Q**2 + U**2 * err_U**2
            ) / pI_safe

            # pI_debiased = np.sqrt(np.maximum(pI_raw**2 - noise_pI**2, 0))
            pI_debiased = pI_raw.copy()

            snr_pI = np.where(
                noise_pI > np.finfo(float).eps,
                pI_debiased / noise_pI,
                0.0
            )
        else:
            noise_pI = np.full_like(I, np.nan, dtype=float)
            pI_debiased = pI_raw.copy()
            snr_pI = np.full_like(I, np.nan, dtype=float)

            # fallback: without errors, SNR masks cannot be physically applied
            # so keep finite good pixels only
            snr_main = -np.inf
            snr_strict = -np.inf

        mask_A = good_mask & np.isfinite(snr_pI) & (snr_pI >= snr_main)
        mask_B = good_mask & np.isfinite(snr_pI) & (snr_pI >= snr_strict)

        I_for_thr = I[good_mask & np.isfinite(I)]

        if I_for_thr.size > 0:
            I_thr = np.nanpercentile(I_for_thr, bright_percentile)
        else:
            I_thr = np.nan

        mask_C = mask_A & np.isfinite(I) & (I >= I_thr)

        masks = {
            "A_reliable_snr2": mask_A,
            "B_strict_snr3": mask_B,
            "C_brightI_snr2": mask_C,
        }

        aux = {
            "pI_raw": pI_raw,
            "pI_debiased": pI_debiased,
            "noise_pI": noise_pI,
            "snr_pI": snr_pI,
            "snr_definition": "P_raw / sigma_P, no debiasing",
            "good_mask": good_mask,
            "I_bright_threshold": I_thr,
            "bright_percentile": bright_percentile,
            "N_good": int(np.sum(good_mask)),
            "N_A": int(np.sum(mask_A)),
            "N_B": int(np.sum(mask_B)),
            "N_C": int(np.sum(mask_C)),
        }

        return masks, aux
    
    def build_fireplace_masks_from_products(
        self,
        I,
        Q,
        U,
        pol_flux,
        err_pol_flux,
        deb_pol_flux=None,
        image_mask=None,
        snr_main=2.0,
        snr_strict=3.0,
        bright_percentile=75,
        snr_mode="raw",   # "raw" = P/sigmaP, "debiased" = P_db/sigmaP
    ):
        import numpy as np

        finite_mask = (
            np.isfinite(I)
            & np.isfinite(Q)
            & np.isfinite(U)
            & np.isfinite(pol_flux)
            & np.isfinite(err_pol_flux)
            & (err_pol_flux > 0)
        )

        if image_mask is not None:
            # FIREPLACE IMAGE MASK is a positive-valued coverage/weight-like map.
            # Positive finite values correspond to valid observed pixels.
            good_mask = finite_mask & np.isfinite(image_mask) & (image_mask > 0)
        else:
            good_mask = finite_mask

        if snr_mode == "debiased" and deb_pol_flux is not None:
            P_for_snr = deb_pol_flux
            snr_definition = "DEBIASED POL FLUX / ERROR POL FLUX"
        else:
            P_for_snr = pol_flux
            snr_definition = "POL FLUX / ERROR POL FLUX"

        snr_pI = np.full_like(I, np.nan, dtype=float)
        valid_snr = good_mask & np.isfinite(P_for_snr) & np.isfinite(err_pol_flux) & (err_pol_flux > 0)
        snr_pI[valid_snr] = P_for_snr[valid_snr] / err_pol_flux[valid_snr]

        mask_A = good_mask & np.isfinite(snr_pI) & (snr_pI >= snr_main)
        mask_B = good_mask & np.isfinite(snr_pI) & (snr_pI >= snr_strict)

        I_for_thr = I[good_mask & np.isfinite(I)]

        if I_for_thr.size > 0:
            I_thr = np.nanpercentile(I_for_thr, bright_percentile)
        else:
            I_thr = np.nan

        mask_C = mask_A & np.isfinite(I) & (I >= I_thr)

        masks = {
            "A_reliable_snr2": mask_A,
            "B_strict_snr3": mask_B,
            "C_brightI_snr2": mask_C,
        }

        aux = {
            "snr_pI": snr_pI,
            "good_mask": good_mask,
            "I_bright_threshold": I_thr,
            "bright_percentile": bright_percentile,
            "snr_definition": snr_definition,
            "N_good": int(np.sum(good_mask)),
            "N_A": int(np.sum(mask_A)),
            "N_B": int(np.sum(mask_B)),
            "N_C": int(np.sum(mask_C)),
        }

        return masks, aux

    def plot_polarization_map(self, I: np.ndarray, vx: np.ndarray, vy: np.ndarray, wcs: WCS, object_name: str, step: int = 4, save_path: str = None):
        """
        Plots the classic dust intensity map overlaid with magnetic field vectors.
        
        Args:
            I (np.ndarray): Stokes I map (background).
            vx (np.ndarray): B-field vector X component.
            vy (np.ndarray): B-field vector Y component.
            wcs (WCS): World Coordinate System for projection.
            object_name (str): Target name for the title.
            step (int): Decimation factor for the vector grid.
        """
        interval = ZScaleInterval()
        vmin, vmax = interval.get_limits(I)
        
        x = np.linspace(0, vx.shape[1] - 1, vx.shape[1])
        y = np.linspace(0, vx.shape[0] - 1, vx.shape[0])
        px, py = np.meshgrid(x, y)

        jcmtmap_mask = np.ones(I.shape)
        #jcmtmap_mask[I < vmax * 0.6] = np.nan

        fig = plt.figure(figsize=(7, 6))
        ax = fig.add_subplot(111, projection=wcs)
        im = ax.imshow(I, origin="lower", vmin=vmin, vmax=vmax, cmap='viridis')
        
        ax.quiver(
            px[::step, ::step] * jcmtmap_mask[::step, ::step],
            py[::step, ::step] * jcmtmap_mask[::step, ::step],
            vx[::step, ::step] * jcmtmap_mask[::step, ::step],
            vy[::step, ::step] * jcmtmap_mask[::step, ::step],
            angles='xy', scale_units='xy', scale=0.2,
            headwidth=0, headlength=0, headaxislength=0,
            pivot="mid", width=0.003, color="k"
        )

        plt.colorbar(im, ax=ax, pad=0.02, label=r"Stokes $I$")
        ax.set_xlabel("RA (J2000)")
        ax.set_ylabel("Dec (J2000)")
        ax.set_title(rf"{object_name} -- Magnetic Field Morphology")
        if save_path:
            fig.savefig(save_path, bbox_inches='tight')
            plt.close(fig)
        else:
            plt.show()
        
    def plot_snr_colored_polarization_map(self, I: np.ndarray, snr_map: np.ndarray, 
                                          vx: np.ndarray, vy: np.ndarray, wcs: WCS, 
                                          object_name: str, step: int = 4, save_path: str = None):
        """
        Plots a B-field map where vectors are colored by their SNR levels:
        1 < SNR <= 3 (cyan), 3 < SNR <= 5 (red), SNR > 5 (magenta).
        """
        vmin, vmax = self._get_robust_limits(I)
        
        x = np.linspace(0, vx.shape[1] - 1, vx.shape[1])
        y = np.linspace(0, vx.shape[0] - 1, vx.shape[0])
        px, py = np.meshgrid(x, y)

        intensity_mask = np.ones(I.shape)
        #intensity_mask[I < vmax * 0.6] = np.nan

        fig = plt.figure(figsize=(7, 6))
        ax = fig.add_subplot(111, projection=wcs)
        im = ax.imshow(I, origin="lower", vmin=vmin, vmax=vmax, cmap='viridis')

        # Цвета: бирюзовый вместо оранжевого
        levels = [2, 3, 5]
        colors = ['magenta', 'red', 'cyan']
        # Понятные математические подписи
        labels = [fr'{levels[0]} $<$ SNR $\leq$ {levels[1]}', fr'{levels[1]} $<$ SNR $\leq$ {levels[2]}', fr'SNR $>$ {levels[2]}']

        for i in range(len(levels)):
            lower = levels[i]
            upper = levels[i+1] if i+1 < len(levels) else np.inf
            
            snr_mask = np.where((snr_map > lower) & (snr_map <= upper), 1.0, np.nan)
            combined_mask = intensity_mask * snr_mask
            
            ax.quiver(
                px[::step, ::step] * combined_mask[::step, ::step],
                py[::step, ::step] * combined_mask[::step, ::step],
                vx[::step, ::step] * combined_mask[::step, ::step],
                vy[::step, ::step] * combined_mask[::step, ::step],
                angles='xy', scale_units='xy', scale=0.2,
                headwidth=0, headlength=0, headaxislength=0,
                pivot="mid", width=0.004, color=colors[i]
            )

        # Легенда с использованием прокси-артистов
        legend_elements = [Line2D([0], [0], color=c, lw=2, label=l) for c, l in zip(colors, labels)]
        ax.legend(handles=legend_elements, loc='upper right', fontsize=9, framealpha=0.8, edgecolor='black')

        plt.colorbar(im, ax=ax, pad=0.02, label=r"Stokes $I$")
        ax.set_xlabel("RA (J2000)")
        ax.set_ylabel("Dec (J2000)")
        ax.set_title(rf"{object_name.replace('_', ' ')} -- SNR-Coded B-Field")
        
        if save_path:
            fig.savefig(save_path, bbox_inches='tight')
            plt.close(fig)
        else:
            plt.show()
            
    def plot_structure_function(self, sf_data: Dict[str, np.ndarray], save_path: str = None):
        """
        Plots the raw and binned polarization angle structure function.
        Uses R_pc if available, otherwise falls back to R in pixels.
        """
        if "R_pc" in sf_data:
            R = sf_data["R_pc"]
            xlabel = r"Separation $R$ (pc)"
        else:
            R = sf_data["R"]
            xlabel = r"Separation $R$ (pix)"

        D = sf_data["Dphi"]
        N = sf_data["Npairs"]

        m = np.isfinite(D) & (N > 0) & np.isfinite(R) & (R > 0)
        Rv, Dv, Nv = R[m], D[m], N[m]

        if len(Rv) == 0:
            print("[Warning] No valid data for structure function plot. Skipping...")
            return

        sig = Dv / np.sqrt(Nv)

        def logbin_median(x, y, nbins=12):
            lx = np.log10(x)
            edges = np.linspace(lx.min(), lx.max(), nbins + 1)
            xb, yb = [], []
            for a, b in zip(edges[:-1], edges[1:]):
                sel = (lx >= a) & (lx < b)
                if np.any(sel):
                    xb.append(10 ** np.median(lx[sel]))
                    yb.append(np.median(y[sel]))
            return np.array(xb), np.array(yb)

        Rb_s, Db_s = logbin_median(Rv, Dv, nbins=10)

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(Rv, Dv, marker="o", ms=4, lw=1, alpha=0.6, color="gray", label=r"Raw $D_\phi(R)$")
        ax.errorbar(Rv, Dv, yerr=sig, fmt="none", ecolor="gray", lw=0.8, alpha=0.5)

        if len(Rb_s) > 1:
            ax.plot(Rb_s, Db_s, marker="s", ms=6, lw=2, color="black", label=r"Log-binned Median")

        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel(xlabel)
        ax.set_ylabel(r"Structure Function $D_\phi(R)$ (rad$^2$)")
        ax.set_title(r"Polarization Angle Dispersion")

        ax.minorticks_on()
        ax.grid(True, which="major", lw=0.8, alpha=0.3)
        ax.grid(True, which="minor", lw=0.4, alpha=0.15)
        ax.legend(frameon=True, edgecolor="black")

        if save_path:
            fig.savefig(save_path, bbox_inches="tight")
            plt.close(fig)
        else:
            plt.show()
            
            
    def plot_snr_distributions(self, p_frac: np.ndarray, psi_deg: np.ndarray, snr_pI: np.ndarray, object_name: str, save_path: Optional[str] = None):
        """
        Plots histograms of Polarization Fraction and EVPA for different 
        SNR thresholds. Essential for analyzing Ricean bias 
        and the statistical ordering of the magnetic field.
        """
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
        
        thresholds = [2, 3, 5]
        colors = ['gray', 'royalblue', 'crimson']
        alphas = [0.4, 0.7, 0.9]
        
        for thr, col, alp in zip(thresholds, colors, alphas):
            mask = (snr_pI > thr) & np.isfinite(p_frac) & np.isfinite(psi_deg)
            valid_p = p_frac[mask]
            valid_psi = psi_deg[mask]
            
            if len(valid_p) == 0:
                continue
                
            # 1. Polarization Fraction Distribution
            bins_p = np.linspace(0, 0.3, 40) # Capped at 30% for visibility
            ax1.hist(valid_p, bins=bins_p, histtype='step', lw=2, 
                     color=col, alpha=alp, density=True, 
                     label=rf'$\sigma > {thr}$ ($N={len(valid_p)}$)')
            
            # 2. EVPA Distribution
            bins_psi = np.linspace(-90, 90, 36)
            ax2.hist(valid_psi, bins=bins_psi, histtype='step', lw=2, 
                     color=col, alpha=alp, density=True, 
                     label=rf'$\sigma > {thr}$')

        ax1.set_xlabel(r"Polarization Fraction $p$")
        ax1.set_ylabel("Probability Density")
        ax1.legend(frameon=False)
        ax1.grid(True, which="major", alpha=0.3)

        ax2.set_xlabel(r"EVPA $\psi$ (deg)")
        ax2.set_ylabel("Probability Density")
        ax2.legend(frameon=False)
        ax2.grid(True, which="major", alpha=0.3)

        fig.suptitle(rf"{object_name.replace('_', ' ')} -- Distributions by Polarized SNR")
        fig.tight_layout()
        
        if save_path:
            for fmt in ['pdf', 'png']:
                fig.savefig(f"{save_path}.{fmt}", bbox_inches='tight')
                plt.close(fig)
        else:
            plt.show()

    def plot_fireplace_masks(
        self,
        I,
        wcs,
        object_name,
        masks,
        aux,
        save_path=None,
    ):
        import numpy as np
        import matplotlib.pyplot as plt
        from astropy.visualization import ZScaleInterval, ImageNormalize, AsinhStretch

        fig = plt.figure(figsize=(14, 4))

        norm = ImageNormalize(
            I,
            interval=ZScaleInterval(),
            stretch=AsinhStretch(a=0.1)
        )

        mask_items = [
            ("A_reliable_snr2", "Mask A: SNR_PI ≥ 2"),
            ("B_strict_snr3", "Mask B: SNR_PI ≥ 3"),
            ("C_brightI_snr2", f"Mask C: SNR_PI ≥ 2 + I ≥ P{aux['bright_percentile']}"),
        ]

        for i, (key, title) in enumerate(mask_items, start=1):
            ax = fig.add_subplot(1, 3, i, projection=wcs)

            im = ax.imshow(I, origin="lower", cmap="viridis", norm=norm)

            m = masks[key]

            ax.contour(
                m.astype(float),
                levels=[0.5],
                colors="red",
                linewidths=1.2,
                transform=ax.get_transform("pixel"),
            )

            ax.set_title(f"{title}\nN={np.sum(m)}")
            ax.set_xlabel("RA")
            ax.set_ylabel("Dec")

        cbar = fig.colorbar(im, ax=fig.axes, fraction=0.025, pad=0.02)
        cbar.set_label("Stokes I")

        fig.suptitle(f"{object_name}: FIREPLACE mask comparison", y=1.02)
        # fig.tight_layout()

        if save_path is not None:
            fig.savefig(save_path, dpi=250, bbox_inches="tight")
            plt.close(fig)
        else:
            plt.show()

    def plot_structure_function_comparison(self, sf_results, save_path=None):
        import numpy as np
        import matplotlib.pyplot as plt
    
        labels = {
            "A_reliable_snr2": "A: reliable region, SNR_PI ≥ 2",
            "B_strict_snr3": "B: strict polarization, SNR_PI ≥ 3",
            "C_brightI_snr2": "C: bright I subset, SNR_PI ≥ 2",
        }
    
        fig, ax = plt.subplots(figsize=(8, 5))
    
        use_pc = any("R_pc" in sf_data for sf_data in sf_results.values())
    
        for key, sf_data in sf_results.items():
            R = sf_data["R_pc"] if "R_pc" in sf_data else sf_data["R"]
            D = sf_data["Dphi"]
            N = sf_data["Npairs"]
    
            m = np.isfinite(R) & np.isfinite(D) & (R > 0) & (N > 0)
    
            if np.sum(m) == 0:
                continue
            
            ax.plot(
                R[m],
                D[m],
                marker="o",
                ms=3,
                lw=1.3,
                alpha=0.8,
                label=labels.get(key, key),
            )
    
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("Separation R (pc)" if use_pc else "Separation R (pix)")
        ax.set_ylabel("Structure Function D_phi(R)")
        ax.set_title("Polarization Angle Structure Function: mask comparison")
        ax.grid(True, which="major", alpha=0.3)
        ax.grid(True, which="minor", alpha=0.15)
        ax.legend(frameon=True, fontsize=8)
    
        fig.tight_layout()
    
        if save_path is not None:
            fig.savefig(save_path, dpi=250, bbox_inches="tight")
            plt.close(fig)
        else:
            plt.show()

    def detect_hills_statistically(self, I, wcs, object_name, var_I=None, qual_I=None, margin_ratio=0.15, sigma_level=2.0, save_path=None):
        import cv2
        import numpy as np
        import matplotlib.pyplot as plt
        from astropy.visualization import ZScaleInterval, ImageNormalize, AsinhStretch
        from scipy.ndimage import distance_transform_edt
        from astropy.stats import sigma_clipped_stats

        ny, nx = I.shape
        
        # --- 1. ГЕОМЕТРИЯ (Ваш "забор") ---
        footprint = np.isfinite(I) & (I != 0)
        dist_map = distance_transform_edt(footprint)
        threshold_dist = np.nanmax(dist_map) * margin_ratio
        safe_zone_mask = dist_map >= threshold_dist

        # --- 2. ЛОГИКА МАСКИРОВАНИЯ ---
        has_quality = (qual_I is not None)
        
        if has_quality:
            # РЕЖИМ А: ПАСПОРТ КАЧЕСТВА (Игнорируем sigma_level)
            # Берем только "хорошие" пиксели (значение 0) внутри забора
            cloud_mask = (qual_I == 0) & safe_zone_mask
            
            method_str = "Hardware QUALITY Mask"
            title_str = fr"{object_name}: clumps (via QUALITY flags)"
            
        else:
            # РЕЖИМ Б: СТАТИСТИЧЕСКАЯ ЭВРИСТИКА (Ваш алгоритм с sigma_level)
            data_in_zone = I[safe_zone_mask & np.isfinite(I)]
            
            if data_in_zone.size == 0:
                print(f"[WARN] Safe zone is empty for {object_name}")
                return [], np.full_like(I, np.inf), np.zeros_like(I), {}

            median_val = np.nanmedian(data_in_zone)
            std_val = np.nanstd(data_in_zone)
            
            # Порог: медиана + N сигм
            science_threshold = median_val + sigma_level * std_val
            cloud_mask = (I > science_threshold) & safe_zone_mask
            
            method_str = f"{sigma_level}$\\sigma$ Heuristic"
            title_str = fr"{object_name}: clumps ($>${sigma_level}$\sigma$ above median)"

        # --- 3. ПОИСК КОНТУРОВ (OpenCV) ---
        cloud_mask_uint8 = cloud_mask.astype('uint8') * 255
        contours, _ = cv2.findContours(cloud_mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # Отсеиваем мелкий мусор сразу, чтобы не тащить его в физику
        valid_contours = [cnt for cnt in contours if cv2.contourArea(cnt) > 12]

        # --- 4. РАСЧЕТ ОШИБОК И ФОНА ---
        # Создаем маску найденных облаков, чтобы вырезать их из фона
        exclusion_mask = np.zeros_like(cloud_mask_uint8)
        for cnt in valid_contours:
            cv2.drawContours(exclusion_mask, [cnt], -1, 255, thickness=cv2.FILLED)
            
        # Честный фон: внутри забора, но вне облаков
        bg_mask = safe_zone_mask & (exclusion_mask == 0)
        bg_data = I[bg_mask & np.isfinite(I)]
        
        if bg_data.size > 0:
            mean_bg, median_bg, std_bg = sigma_clipped_stats(bg_data, sigma=3.0)
        else:
            mean_bg, median_bg, std_bg = np.nan, np.nan, np.nanstd(I)

        # Формируем карту ошибок для физики векторов
        if var_I is not None:
            # Если есть VARIANCE, используем его для SNR векторов (даже если нет QUALITY)
            err_I_map = np.sqrt(np.where(var_I > 0, var_I, np.inf))
            if has_quality:
                err_I_map[qual_I != 0] = np.inf
            snr_map = np.where(err_I_map < np.inf, I / err_I_map, 0)
            if not has_quality:
                method_str += " + Hardware Variance"
        else:
            # Если аппаратных ошибок вообще нет, полагаемся на фон
            err_I_map = np.full_like(I, std_bg)
            err_I_map[~np.isfinite(I)] = np.inf
            snr_map = np.where(std_bg > 0, I / std_bg, 0)

        # --- 5. ОТРИСОВКА ---
        fig = plt.figure(figsize=(8, 7))
        ax = fig.add_subplot(111, projection=wcs)
        
        norm = ImageNormalize(I, interval=ZScaleInterval(), stretch=AsinhStretch(a=0.1))
        im = ax.imshow(I, origin='lower', cmap='viridis', norm=norm)

        # Рисуем забор пунктиром
        ax.contour(safe_zone_mask, levels=[0.5], colors='magenta', linewidths=1.2, alpha=0.8, linestyles='dashed')

        # Рисуем финальные контуры
        count = len(valid_contours)
        for cnt in valid_contours:
            poly = cnt.reshape(-1, 2)
            poly = np.vstack([poly, poly[0]])
            ax.plot(poly[:, 0], poly[:, 1], color='red', lw=1.5)

        # Обновляем заголовок с актуальным количеством объектов
        final_title = title_str.replace("clumps", f"{count} clumps")
        ax.set_title(final_title)
        
        plt.colorbar(im, ax=ax, pad=0.02, label=r"Stokes $I$")

        if save_path:
            plt.savefig(str(save_path), bbox_inches='tight')
            plt.close(fig)
        else:
            plt.show()

        stats_dict = {'mean_bg': mean_bg, 'median_bg': median_bg, 'std_bg': std_bg, 'method': method_str}
        return valid_contours, err_I_map, snr_map, stats_dict
    
    
    def load_stokes_maps_with_variance(self, object_name: str, band: str = "850um"):
        """
        Гибкий поиск файлов: ищет вхождения (object_name, parameter, 'map') 
        в любом регистре и порядке.
        """
        from astropy.io import fits
        from astropy.wcs import WCS
        import numpy as np
        import os

        maps_dir = self.base_dir / object_name / "maps"
        if not maps_dir.exists():
            raise FileNotFoundError(f"Директория не найдена: {maps_dir}")

        # Подготовка "отпечатка" объекта: нижний регистр, без дефисов и подчеркиваний
        clean_obj = object_name.lower().replace('_', '').replace('-', '')
        all_files = [f for f in os.listdir(maps_dir) if f.lower().endswith(('.fits', '.fit'))]

        def find_file(stokes_char):
            s_low = stokes_char.lower()
            # Защита от поиска одинокой буквы внутри других слов (например, 'i' в 'auriga')
            patterns = [f"_{s_low}_", f"{s_low}map", f"{s_low}_map"]
            
            for f in all_files:
                f_low = f.lower()
                # Тотальная зачистка имени файла для безопасного сравнения объекта
                f_clean_name = f_low.replace('_', '').replace('-', '')
                
                is_obj = clean_obj in f_clean_name
                has_stokes = any(p in f_low for p in patterns)
                
                if is_obj and has_stokes and "map" in f_low:
                    return maps_dir / f
            return None

        # Ищем пути для I, Q, U
        found_paths = {p: find_file(p) for p in ['I', 'Q', 'U']}

        # Проверка на вылет
        for p, path in found_paths.items():
            if path is None:
                print(f"[ERROR] Не удалось найти файл для параметра {p} в {maps_dir}")
                print(f"Искали: {clean_obj} + {p.lower()} + 'map'")
                raise FileNotFoundError(f"Missing {p} map for {object_name}")

        def extract_fits_layers(filepath):
            with fits.open(filepath) as hdul:
                # HDU 0: Основные данные
                data = np.squeeze(hdul[0].data)
                # HDU 1: VARIANCE (если есть)
                var_data = np.squeeze(hdul[1].data) if len(hdul) > 1 else None
                # HDU 2: QUALITY (если есть)
                qual_data = np.squeeze(hdul[2].data) if len(hdul) > 2 else None
                return data, var_data, qual_data

        # Извлечение данных с использованием найденных путей
        I, var_I, qual_I = extract_fits_layers(found_paths['I'])
        Q, var_Q, qual_Q = extract_fits_layers(found_paths['Q'])
        U, var_U, qual_U = extract_fits_layers(found_paths['U'])

        # Чтение WCS из найденного файла I
        with fits.open(found_paths['I']) as hdul_I:
            wcs = WCS(hdul_I[0].header).celestial 

        return I, Q, U, wcs, var_I, var_Q, var_U, qual_I, qual_Q, qual_U

    def build_sf_table(
        self,
        sf_data: Dict[str, np.ndarray],
        object_name: Optional[str] = None,
        mask_name: Optional[str] = None,
        distance_pc: Optional[float] = None,
    ) -> pd.DataFrame:
        """ 
        Convert structure-function output dictionary into a tabular form.

        Required keys in sf_data:
        - R
        - Dphi
        - Npairs

        Optional keys:
        - R_pc
        """
        required = ["R", "Dphi", "Npairs"]
        for key in required:
            if key not in sf_data:
                raise KeyError(f"sf_data is missing required key '{key}'.")

        table = {
            "R_pix": np.asarray(sf_data["R"], dtype=float),
            "Dphi": np.asarray(sf_data["Dphi"], dtype=float),
            "Npairs": np.asarray(sf_data["Npairs"]),
        }

        if "R_pc" in sf_data:
            table["R_pc"] = np.asarray(sf_data["R_pc"], dtype=float)

        df = pd.DataFrame(table)

        if object_name is not None:
            df.insert(0, "object_name", object_name)

        if mask_name is not None:
            insert_at = 1 if object_name is not None else 0
            df.insert(insert_at, "mask_name", mask_name)

        if distance_pc is not None:
            df["distance_pc"] = distance_pc

        return df


    def save_sf_table(
        self,
        sf_data: Dict[str, np.ndarray],
        save_path: str | Path,
        object_name: Optional[str] = None,
        mask_name: Optional[str] = None,
        distance_pc: Optional[float] = None,
    ) -> Path:
        """
        Save structure-function data to CSV.
        """
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)

        df = self.build_sf_table(
            sf_data=sf_data,
            object_name=object_name,
            mask_name=mask_name,
            distance_pc=distance_pc,
        )
        df.to_csv(save_path, index=False)
        return save_path
    
    
    def process_object(self, object_name: str, distance_pc: float, band: str = "850um", margin_ratio=0.08, sigma_level=1.5, step: int = 2):

        print(f"\n{'='*60}\nPROCESSING: {object_name}\n{'='*60}")

        try:
            # ОЖИДАЕМАЯ СИГНАТУРА ЗАГРУЗКИ:
            # Если var_* или qual_* файлов нет, функция загрузки должна возвращать None на их местах.
            I, Q, U, wcs, var_I, var_Q, var_U, qual_I, qual_Q, qual_U = self.load_stokes_maps_with_variance(object_name, band)
        except FileNotFoundError:
            return None, None, None

        report_dir = self.base_dir.parents[1] / "report"
        plots_dir = report_dir / "plots" / object_name
        plots_dir.mkdir(parents=True, exist_ok=True)

        # --- ЭТАП 1: СЕГМЕНТАЦИЯ СТРУКТУР ---
        diag_path = plots_dir / f"{object_name}_stat_segmentation.pdf"
        contours, err_I_map, snr_I_map, stats_I = self.detect_hills_statistically(
            I, wcs, object_name, 
            var_I=var_I, qual_I=qual_I,
            margin_ratio=margin_ratio, 
            sigma_level=sigma_level, 
            save_path=str(diag_path)
        )

        if not contours:
            print(f"[ERROR] No clumps found for {object_name}. Physics is canceled.")
            return None, None, None

        # --- ЭТАП 2: ФИЗИЧЕСКАЯ ИЗОЛЯЦИЯ ---
        science_mask = np.zeros(I.shape, dtype=np.uint8)
        for cnt in contours:
            if cv2.contourArea(cnt) > 12: 
                cv2.drawContours(science_mask, [cnt], -1, 1, thickness=cv2.FILLED)

        # Зачистка битых пикселей I
        if qual_I is not None:
            science_mask[qual_I != 0] = 0

        science_mask_bool = science_mask.astype(bool)

        I_full = I.copy()
        I[~science_mask_bool] = np.nan
        Q[~science_mask_bool] = np.nan
        U[~science_mask_bool] = np.nan

        # --- ЭТАП 3: СТРОГАЯ ПРОПАГАЦИЯ ОШИБОК И ФИЗИКА ---
        def get_stokes_error_map(data, var_map, qual_map, science_mask_bool):
            """Внутренний Fallback для карт Q и U"""
            # ИСПРАВЛЕНИЕ: Проверяем только var_map
            if var_map is not None:
                err = np.sqrt(np.where(var_map > 0, var_map, np.inf))
                if qual_map is not None:
                    err[qual_map != 0] = np.inf
                return err
            else:
                # Эмпирический расчет по фону
                bg_data = data[~science_mask_bool & np.isfinite(data)]
                from astropy.stats import sigma_clipped_stats
                std_val = sigma_clipped_stats(bg_data, sigma=3.0)[2] if bg_data.size > 0 else np.nanstd(data)
                return np.full_like(data, std_val)

        # 1. Извлекаем ошибки Q и U
        err_Q = get_stokes_error_map(Q, var_Q, qual_Q, science_mask_bool)
        err_U = get_stokes_error_map(U, var_U, qual_U, science_mask_bool)

        # 2. Интенсивность поляризации (Сырая)
        pI_raw = np.hypot(Q, U)

        # 3. Ошибка интенсивности поляризации (Аналитическая)
        pI_safe = np.where(pI_raw > 0, pI_raw, np.inf)
        noise_pI = np.sqrt(Q**2 * err_Q**2 + U**2 * err_U**2) / pI_safe

        # 4. ДЕБИАСИНГ (Коррекция Уордла-Кронберга)
        radicand = pI_raw**2 - noise_pI**2
        pI_debiased = np.sqrt(np.maximum(radicand, 0)) # np.maximum изящнее чем np.where

        # 5. Итоговый SNR для векторов (по дебиасированному PI)
        snr_pI_map = np.where(noise_pI > np.finfo(float).eps, pI_debiased / noise_pI, 0)

        # 6. Жесткий фильтр для структурных функций (оставляем только достоверные векторы)
        # Пиксели с SNR_PI < 3 станут NaN
        valid_vectors = (snr_pI_map >= 2.0)
        Q[~valid_vectors] = np.nan
        U[~valid_vectors] = np.nan

        # Ошибка угла (для взвешивания в структурных функциях, опционально)
        # noise_psi_rad = (1.0 / (2.0 * pI_safe**2)) * np.sqrt(Q**2 * err_U**2 + U**2 * err_Q**2)

        psi_deg = self.compute_evpa(Q, U)
        vx, vy = self.pol_vector_components(p=1.0, psi_rad=np.radians(psi_deg))
        scales = self.get_physical_scales(wcs, distance_pc, I_full.shape)

    # --- ЭТАП 4: ГРАФИКА И ВЫВОД ---
        ny, nx = I.shape
        max_R = int(min(ny, nx) * (2 / 3))
        R_list_px = np.arange(1, max_R + 1)

        # Расчет структурной функции
        sf_data = self.compute_structure_function(Q, U, R_list=R_list_px, min_pI=None)

        if sf_data is not None:
            sf_data["R_pc"] = R_list_px * scales["phys_scale_pc"]
            self.plot_structure_function(sf_data, save_path=plots_dir/f"{object_name}_sf.pdf")

        # Отрисовка карт
        self.plot_polarization_map(I_full, vx, vy, wcs, f"{object_name} (Full Context)", step, 
                                   save_path=plots_dir/f"{object_name}_pol_map_full.pdf")
        self.plot_polarization_map(I, vx, vy, wcs, f"{object_name} (Isolated Cores)", step, 
                                   save_path=plots_dir/f"{object_name}_pol_map_cropped.pdf")
        self.plot_snr_colored_polarization_map(I_full, snr_pI_map, vx, vy, wcs, f"{object_name} (Full Context)", step, 
                                               save_path=plots_dir/f"{object_name}_snr_colored_full.pdf")
        self.plot_snr_colored_polarization_map(I, snr_pI_map, vx, vy, wcs, f"{object_name} (Isolated Cores)", step, 
                                               save_path=plots_dir/f"{object_name}_snr_colored_cropped.pdf")

        # --- ЭТАП 5: ГЕНЕРАЦИЯ ОТЧЕТА ---
        valid_count = np.sum(valid_vectors & science_mask_bool)
        median_noise_I = np.nanmedian(err_I_map[science_mask_bool])

        # Вычисляем физический предел R для отчета
        r_min_pc = 1 * scales['phys_scale_pc']
        r_max_pc = max_R * scales['phys_scale_pc']

        # Определяем, какой метод использовался для Q и U
        method_QU = "Hardware Variance" if (var_Q is not None and var_U is not None) else f"Heuristic (Background RMS)"
        median_snr_pi = np.nanmedian(snr_pI_map[valid_vectors & science_mask_bool]) if valid_count > 0 else 0

        report = f"""
            ============================================================
            [+] OBJECT PASSPORT: {object_name}
            ============================================================
            > NOISE ESTIMATION METHODOLOGY
              - Stokes parameter I:    {stats_I['method']}
              - Stokes vectors Q/U:   {method_QU}
              - PI debiasing:         Applied (Wardle & Kronberg 1974)

            > GEOMETRY AND SCALES
              - Adopted distance:     {distance_pc} pc
              - Lag range (R):        from 1 to {max_R} pixels
                                      ({r_min_pc:.4f} pc — {r_max_pc:.4f} pc)
              - Pixel scale:          {scales['phys_scale_pc']:.5f} pc/pixel

            > STATISTICS AND PHYSICAL SAMPLE
              - Effective area:       {valid_count} pixels (SNR_PI > 2.0)
              - Median noise (I):     {median_noise_I:.4f} mJy/beam
              - Median SNR (PI):      {median_snr_pi:.2f}
            ============================================================
            """
        print(report)

        # Упаковываем данные для возврата, если они нужны внешним скриптам
        stats = {
            "Distance (pc)": distance_pc,
            "R_range_px": (1, max_R),
            "R_range_pc": (r_min_pc, r_max_pc),
            "Effective Pixels": valid_count,
            "Method I": stats_I['method'],
            "Method QU": method_QU
        }
        # --- ЭТАП 6: СОХРАНЕНИЕ СТАТИСТИКИ ---
        # Формируем путь: report/plots/ObjectName/ObjectName_stats.json
        stats_path = plots_dir / f"{object_name}_stats.json"
        
        # Подготавливаем словарь (убеждаемся, что все типы данных сериализуемы)
        serializable_stats = {}
        for k, v in stats.items():
            if isinstance(v, (np.ndarray, np.generic)):
                serializable_stats[k] = v.tolist()
            else:
                serializable_stats[k] = v

        with open(stats_path, "w", encoding="utf-8") as f:
            json.dump(serializable_stats, f, indent=4, ensure_ascii=False)
            
        print(f"[DISK] Statistics saved to {stats_path}")

        return psi_deg, sf_data, stats
    
    
    
    def process_fireplace_cutout(
        self,
        cutout_path,
        output_dir,
        distance_pc=None,
        step=4,
        bright_percentile=75,
        snr_main=2.0,
        snr_strict=3.0,
    ):
        from pathlib import Path
        import numpy as np
        import pandas as pd
        import json
    
        cutout_path = Path(cutout_path)
        output_dir = Path(output_dir)
    
        print(f"\n{'='*60}")
        print(f"PROCESSING FIREPLACE CUTOUT: {cutout_path.name}")
        print(f"{'='*60}")
    

        data = self.load_fireplace_cutout(cutout_path)

        for key in ["I", "Q", "U", "pol_flux", "err_pol_flux", "deb_pol_flux", "mask"]:
            arr = data.get(key)
            if arr is None:
                print(key, "None")
            else:
                print(key, arr.shape, "finite:", np.sum(np.isfinite(arr)))

        object_name = data["object_name"]
        I = data["I"]
        Q = data["Q"]
        U = data["U"]
        wcs = data["wcs"]

        plots_dir = output_dir / "plots" / object_name
        plots_dir.mkdir(parents=True, exist_ok=True)

        masks, aux = self.build_fireplace_masks_from_products(
            I=I,
            Q=Q,
            U=U,
            pol_flux=data["pol_flux"],
            err_pol_flux=data["err_pol_flux"],
            deb_pol_flux=data["deb_pol_flux"],
            image_mask=data["mask"],
            snr_main=snr_main,
            snr_strict=snr_strict,
            bright_percentile=bright_percentile,
            snr_mode="raw",       # для теста P/sigmaP
            # snr_mode="debiased" # потом можно сравнить
        )
    
        self.plot_fireplace_masks(
            I=I,
            wcs=wcs,
            object_name=object_name,
            masks=masks,
            aux=aux,
            save_path=plots_dir / f"{object_name}_mask_comparison.pdf",
        )
    
        if distance_pc is not None:
            scales = self.get_physical_scales(wcs, distance_pc, I.shape)
        else:
            scales = None
    
        ny, nx = I.shape
        max_R = int(min(ny, nx) * (2 / 3))
        R_list_px = np.arange(1, max_R + 1)
    
        sf_results = {}
    
        for mask_name, this_mask in masks.items():
            Q_sf = Q.copy()
            U_sf = U.copy()
        
            Q_sf[~this_mask] = np.nan
            U_sf[~this_mask] = np.nan
        
            sf_data = self.compute_structure_function(Q_sf, U_sf, R_list=R_list_px)

            if scales is not None:
                sf_data["R_pc"] = R_list_px * scales["phys_scale_pc"]
        
            sf_results[mask_name] = sf_data
        
            self.save_sf_table(
                sf_data=sf_data,
                save_path=plots_dir / f"{object_name}_sf_{mask_name}.csv",
                object_name=object_name,
                mask_name=mask_name,
                distance_pc=distance_pc,
            )
        
            self.plot_structure_function(
                sf_data,
                save_path=plots_dir / f"{object_name}_sf_{mask_name}.pdf",
            )
    
        self.plot_structure_function_comparison(
            sf_results,
            save_path=plots_dir / f"{object_name}_sf_mask_comparison.pdf",
        )
    
        # One representative vector map for the main mask A
        main_mask = masks["A_reliable_snr2"]
    
        Q_main = Q.copy()
        U_main = U.copy()
        Q_main[~main_mask] = np.nan
        U_main[~main_mask] = np.nan
    
        psi_deg = self.compute_evpa(Q_main, U_main)
        vx, vy = self.pol_vector_components(
            p=1.0,
            psi_rad=np.radians(psi_deg),
        )
    
        I_plot = I.copy()
        I_plot[~aux["good_mask"]] = np.nan
    
        self.plot_polarization_map(
            I_plot,
            vx,
            vy,
            wcs,
            f"{object_name} | FIREPLACE | Mask A",
            step=step,
            save_path=plots_dir / f"{object_name}_pol_map_maskA.pdf",
        )
    
        self.plot_snr_colored_polarization_map(
            I_plot,
            aux["snr_pI"],
            vx,
            vy,
            wcs,
            f"{object_name} | FIREPLACE | Mask A",
            step=step,
            save_path=plots_dir / f"{object_name}_snr_colored_maskA.pdf",
        )
    
        stats = {
            "file": cutout_path.name,
            "object_name": object_name,
            "distance_pc": distance_pc,
            "ny": int(I.shape[0]),
            "nx": int(I.shape[1]),
            "R_range_px": [1, int(max_R)],
            "bright_percentile": bright_percentile,
            "I_bright_threshold": float(aux["I_bright_threshold"]) if np.isfinite(aux["I_bright_threshold"]) else None,
            "N_good": aux["N_good"],
            "N_mask_A_reliable_snr2": aux["N_A"],
            "N_mask_B_strict_snr3": aux["N_B"],
            "N_mask_C_brightI_snr2": aux["N_C"],
            "method": "FIREPLACE rectangular cutout with three masks: SNR2, SNR3, bright-I subset",
            "PI_debiasing": "Wardle-Kronberg",
            "snr_definition": aux.get("snr_definition", "unknown"),
        }
        
        if scales is not None:
            stats["pixel_scale_pc"] = float(scales["phys_scale_pc"])
            stats["R_range_pc"] = [
                float(scales["phys_scale_pc"]),
                float(max_R * scales["phys_scale_pc"]),
            ]
        else:
            stats["pixel_scale_pc"] = None
            stats["R_range_pc"] = None
    
        stats_path = plots_dir / f"{object_name}_stats.json"
    
        with open(stats_path, "w", encoding="utf-8") as f:
            json.dump(stats, f, indent=4, ensure_ascii=False)
    
        print(f"[OK] Finished {object_name}")
        print(f"[DISK] Saved to: {plots_dir}")
    
        return sf_results, stats

# Implemetation for one objrct

In [42]:
# from pathlib import Path
# import pandas as pd
# import matplotlib as mpl
# import matplotlib.pyplot as plt

# mpl.rcParams.update({
#     "text.usetex": False,
#     "font.family": "DejaVu Sans",
#     "mathtext.fontset": "dejavusans",
# })

# CUTOUT_ROOT = Path("data/FIREPLACE/fireplace_science_region_cutouts")
# OUTPUT_DIR = Path("report/fireplace_results")

# PLOTS_DIR = OUTPUT_DIR / "plots"
# REPORT_DIR = OUTPUT_DIR / "report_text"
# REPORT_DIR.mkdir(parents=True, exist_ok=True)

# cutout_files = sorted(CUTOUT_ROOT.glob("*/*_cutout.fits"))

# analyzer = PolarizationAnalyzer(base_archive_path=CUTOUT_ROOT)

# cutout_path = cutout_files[0]

# results, stats = analyzer.process_fireplace_cutout(
#     cutout_path=cutout_path,
#     output_dir=OUTPUT_DIR,
#     distance_pc=8200,
#     step=4,
#     bright_percentile=75,
#     snr_main=2.0,      # Mask A
#     snr_strict=3.0,    # Mask B
# )

In [43]:
from pathlib import Path
import matplotlib as mpl

mpl.rcParams.update({
    "text.usetex": False,
    "font.family": "DejaVu Sans",
    "mathtext.fontset": "dejavusans",
})

FIELDMAPS_ROOT = Path(r"data/FIELDMAPS_Data_Release/FIELDMAPS_Reprojection/Original")
OUTPUT_DIR = Path("report/fieldmaps_results")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

fits_files = sorted(FIELDMAPS_ROOT.glob("*.fits"))

analyzer = PolarizationAnalyzer(base_archive_path=FIELDMAPS_ROOT)

fits_path = fits_files[0]   # например Fil1_RA_Original.fits

results, stats = analyzer.process_fireplace_cutout(
    cutout_path=fits_path,
    output_dir=OUTPUT_DIR,
    distance_pc=None,  
    step=4,
    bright_percentile=75,
    snr_main=2.0,
    snr_strict=3.0,
)


PROCESSING FIREPLACE CUTOUT: Fil10_RA_Original.fits
I (266, 495) finite: 61512
Q (266, 495) finite: 61512
U (266, 495) finite: 61512
pol_flux (266, 495) finite: 61512
err_pol_flux (266, 495) finite: 61512
deb_pol_flux (266, 495) finite: 61512
mask (266, 495) finite: 131670
[OK] Finished Fil10_Scan1
[DISK] Saved to: report\fieldmaps_results\plots\Fil10_Scan1


In [44]:
# from astropy.io import fits
# import numpy as np

# with fits.open(cutout_path) as hdul:
#     m = hdul["IMAGE MASK"].data

# print("unique mask values:", np.unique(m[np.isfinite(m)]))
# print("N mask == 0:", np.sum(m == 0))
# print("N mask > 0:", np.sum(m > 0))
# print("N finite mask:", np.sum(np.isfinite(m)))

# Implementation for all objs in dir with report in .tex

In [45]:
# all_stats = []

# for cutout_path in cutout_files:
#     results, stats = analyzer.process_fireplace_cutout(
#         cutout_path=cutout_path,
#         output_dir=OUTPUT_DIR,
#         distance_pc=8200,
#         step=4,
#         bright_percentile=75,
#     )

#     if stats is not None:
#         all_stats.append(stats)

# summary = pd.DataFrame(all_stats)
# summary.to_csv(OUTPUT_DIR / "fireplace_cutout_summary.csv", index=False)

# summary

In [46]:
all_stats = []

for cutout_path in fits_files:
    results, stats = analyzer.process_fireplace_cutout(
        cutout_path=cutout_path,
        output_dir=OUTPUT_DIR,
        distance_pc=None,
        step=4,
        bright_percentile=75,
    )

    if stats is not None:
        all_stats.append(stats)

summary = pd.DataFrame(all_stats)
summary.to_csv(OUTPUT_DIR / "fireplace_cutout_summary.csv", index=False)

summary


PROCESSING FIREPLACE CUTOUT: Fil10_RA_Original.fits
I (266, 495) finite: 61512
Q (266, 495) finite: 61512
U (266, 495) finite: 61512
pol_flux (266, 495) finite: 61512
err_pol_flux (266, 495) finite: 61512
deb_pol_flux (266, 495) finite: 61512
mask (266, 495) finite: 131670
[OK] Finished Fil10_Scan1
[DISK] Saved to: report\fieldmaps_results\plots\Fil10_Scan1

PROCESSING FIREPLACE CUTOUT: Fil1_RA_Original.fits
I (226, 165) finite: 23792
Q (226, 165) finite: 23792
U (226, 165) finite: 23792
pol_flux (226, 165) finite: 23792
err_pol_flux (226, 165) finite: 23792
deb_pol_flux (226, 165) finite: 23792
mask (226, 165) finite: 37290
[OK] Finished Fil1_Scan1
[DISK] Saved to: report\fieldmaps_results\plots\Fil1_Scan1

PROCESSING FIREPLACE CUTOUT: Fil2_RA_Original.fits
I (459, 197) finite: 51210
Q (459, 197) finite: 51210
U (459, 197) finite: 51210
pol_flux (459, 197) finite: 51210
err_pol_flux (459, 197) finite: 51210
deb_pol_flux (459, 197) finite: 51210
mask (459, 197) finite: 90423
[OK] Fini

[OK] Finished G18.6_Field_1
[DISK] Saved to: report\fieldmaps_results\plots\G18.6_Field_1

PROCESSING FIREPLACE CUTOUT: Fil5ChopA_RA_Original.fits
I (118, 118) finite: 9117
Q (118, 118) finite: 8644
U (118, 118) finite: 8644
pol_flux (118, 118) finite: 8644
err_pol_flux (118, 118) finite: 8644
deb_pol_flux (118, 118) finite: 8644
mask (118, 118) finite: 13924


[OK] Finished G18_F8
[DISK] Saved to: report\fieldmaps_results\plots\G18_F8

PROCESSING FIREPLACE CUTOUT: Fil8_RA_Original.fits
I (226, 177) finite: 25350
Q (226, 177) finite: 25350
U (226, 177) finite: 25350
pol_flux (226, 177) finite: 25350
err_pol_flux (226, 177) finite: 25350
deb_pol_flux (226, 177) finite: 25350
mask (226, 177) finite: 40002
[OK] Finished Fil8_Scan3_ALT
[DISK] Saved to: report\fieldmaps_results\plots\Fil8_Scan3_ALT

PROCESSING FIREPLACE CUTOUT: G24_RA_Original.fits
I (563, 514) finite: 91742
Q (563, 514) finite: 91742
U (563, 514) finite: 91742
pol_flux (563, 514) finite: 91742
err_pol_flux (563, 514) finite: 91742
deb_pol_flux (563, 514) finite: 91742
mask (563, 514) finite: 289382
[OK] Finished G24_Scan1
[DISK] Saved to: report\fieldmaps_results\plots\G24_Scan1

PROCESSING FIREPLACE CUTOUT: G47_RA_Original.fits
I (344, 168) finite: 39422
Q (344, 168) finite: 39422
U (344, 168) finite: 39422
pol_flux (344, 168) finite: 39422
err_pol_flux (344, 168) finite: 39422


[OK] Finished Snake_F3
[DISK] Saved to: report\fieldmaps_results\plots\Snake_F3


,file,object_name,distance_pc,ny,nx,R_range_px,bright_percentile,I_bright_threshold,N_good,N_mask_A_reliable_snr2,N_mask_B_strict_snr3,N_mask_C_brightI_snr2,method,PI_debiasing,snr_definition,pixel_scale_pc,R_range_pc
0,Fil10_RA_Original.fits,Fil10_Scan1,None,266,495,"[1, 177]",75,0.256666,61512,18963,10219,9759,FIREPLACE rectangular cutout with three masks:...,Wardle-Kronberg,POL FLUX / ERROR POL FLUX,None,None
1,Fil1_RA_Original.fits,Fil1_Scan1,None,226,165,"[1, 110]",75,0.116774,23792,9803,6096,4774,FIREPLACE rectangular cutout with three masks:...,Wardle-Kronberg,POL FLUX / ERROR POL FLUX,None,None
2,Fil2_RA_Original.fits,Fil2_Scan1,None,459,197,"[1, 131]",75,0.133370,51210,25973,16237,10161,FIREPLACE rectangular cutout with three masks:...,Wardle-Kronberg,POL FLUX / ERROR POL FLUX,None,None
3,Fil4_RA_Original.fits,Fil4_Scan1,None,249,216,"[1, 144]",75,0.128003,34236,18510,11951,7473,FIREPLACE rectangular cutout with three masks:...,Wardle-Kronberg,POL FLUX / ERROR POL FLUX,None,None
4,Fil5_RA_Original.fits,Fil5_Scan3,None,340,128,"[1, 85]",75,0.135120,34963,13381,7392,5014,FIREPLACE rectangular cutout with three masks:...,Wardle-Kronberg,POL FLUX / ERROR POL FLUX,None,None
5,Fil5Chop_RA_Original.fits,G18.6_Field_1,None,102,99,"[1, 66]",75,0.336348,5948,4241,3273,1279,FIREPLACE rectangular cutout with three masks:...,Wardle-Kronberg,POL FLUX / ERROR POL FLUX,None,None
6,Fil5ChopA_RA_Original.fits,G18_F8,None,118,118,"[1, 78]",75,0.106610,8641,2629,1427,1471,FIREPLACE rectangular cutout with three masks:...,Wardle-Kronberg,POL FLUX / ERROR POL FLUX,None,None
7,Fil8_RA_Original.fits,Fil8_Scan3_ALT,None,226,177,"[1, 118]",75,0.137904,25350,9911,4709,4114,FIREPLACE rectangular cutout with three masks:...,Wardle-Kronberg,POL FLUX / ERROR POL FLUX,None,None
8,G24_RA_Original.fits,G24_Scan1,None,563,514,"[1, 342]",75,0.177184,91742,45516,29191,18736,FIREPLACE rectangular cutout with three masks:...,Wardle-Kronberg,POL FLUX / ERROR POL FLUX,None,None
9,G47_RA_Original.fits,G47_Scan1,None,344,168,"[1, 112]",75,0.054741,39422,19479,11717,8409,FIREPLACE rectangular cutout with three masks:...,Wardle-Kronberg,POL FLUX / ERROR POL FLUX,None,None


In [47]:
# from pathlib import Path
# import json
# import re

# # =========================================================
# # PATHS
# # =========================================================
# OUTPUT_DIR = Path("report/fireplace_results")
# PLOTS_DIR = OUTPUT_DIR / "plots"
# REPORT_DIR = OUTPUT_DIR / "report_text"
# REPORT_DIR.mkdir(parents=True, exist_ok=True)

# # =========================================================
# # HELPERS
# # =========================================================
# def latex_escape(s):
#     """
#     Escape common LaTeX special characters.
#     """
#     if s is None:
#         return ""
#     s = str(s)
#     replacements = {
#         "\\": r"\textbackslash{}",
#         "&": r"\&",
#         "%": r"\%",
#         "$": r"\$",
#         "#": r"\#",
#         "_": r"\_",
#         "{": r"\{",
#         "}": r"\}",
#         "~": r"\textasciitilde{}",
#         "^": r"\textasciicircum{}",
#     }
#     return "".join(replacements.get(ch, ch) for ch in s)


# def format_value(v):
#     """
#     Format values for LaTeX table.
#     """
#     if isinstance(v, float):
#         return f"{v:.5g}"
#     if isinstance(v, int):
#         return str(v)
#     if isinstance(v, list) or isinstance(v, tuple):
#         if all(isinstance(x, (int, float)) for x in v):
#             return " -- ".join(f"{x:.5g}" for x in v)
#         return latex_escape(v)
#     if isinstance(v, dict):
#         return latex_escape(json.dumps(v))
#     return latex_escape(v)


# def include_if_exists(path_from_tex, caption, width=r"\linewidth"):
#     """
#     Returns LaTeX subfigure block if the target figure exists.
#     path_from_tex is relative to REPORT_DIR.
#     """
#     full_path = REPORT_DIR / path_from_tex
#     if not full_path.exists():
#         return ""

#     return rf"""
#     \begin{{subfigure}}{{0.48\textwidth}}
#         \centering
#         \includegraphics[width={width}]{{{path_from_tex.as_posix()}}}
#         \caption{{{caption}}}
#     \end{{subfigure}}
# """


# # =========================================================
# # METHODOLOGY TEXT
# # =========================================================
# METHODOLOGY_TEXT = r"""
# \section*{Methodology and Data Selection}

# This report summarizes the polarization angle structure-function analysis for FIREPLACE HAWC+ cutouts of CMZ cloud regions. 
# The analysis is based on the merged FIREPLACE polarization map rather than on individual CAL-level observations. 
# Each region is treated as a physically motivated rectangular aperture around a named CMZ cloud or cloud complex.

# \paragraph{Input data.}
# For each region, we use a cutout containing Stokes \(I\), \(Q\), and \(U\), together with the available uncertainty maps and a validity mask when present. 
# The cutouts preserve the local WCS and are used as the main science apertures.

# \paragraph{Polarization and debiasing.}
# The polarized intensity is computed as
# \[
# PI = \sqrt{Q^2 + U^2}.
# \]
# When uncertainty maps are available, the polarized-intensity uncertainty is propagated from the Stokes \(Q\) and \(U\) errors. 
# We apply the Wardle--Kronberg debiasing correction,
# \[
# PI_{\rm db} = \sqrt{PI^2 - \sigma_{PI}^2},
# \]
# with negative values inside the square root clipped to zero.

# \paragraph{Masking strategy.}
# For each region, the structure function is computed using three masks:
# \begin{enumerate}
#     \item \textbf{Mask A}: all reliable polarization detections within the rectangular cloud aperture, using \(SNR_{PI} \geq 2\);
#     \item \textbf{Mask B}: a stricter polarization-quality subset, using \(SNR_{PI} \geq 3\);
#     \item \textbf{Mask C}: a bright-emission subset, using \(SNR_{PI} \geq 2\) and retaining only pixels above a selected percentile of the Stokes \(I\) distribution inside the region.
# \end{enumerate}
# This three-mask approach is used to test how sensitive the structure function is to the adopted region definition and polarization-quality threshold. 
# The bright-emission mask is treated as a robustness test and is not interpreted as an independent physical cloud boundary.

# \paragraph{Structure function.}
# For each mask, we compute the polarization angle structure function \(D_\phi(R)\) as a function of projected separation \(R\). 
# The separation is converted from pixels to parsecs using the WCS pixel scale and an adopted Galactic Center distance.
# """

# # =========================================================
# # LATEX HEADER
# # =========================================================
# latex_content = r"""\documentclass[10pt, a4paper]{article}
# \usepackage[margin=1.5cm]{geometry}
# \usepackage{graphicx}
# \usepackage{booktabs}
# \usepackage{subcaption}
# \usepackage{float}
# \usepackage{amsmath}
# \usepackage{mathpazo}
# \usepackage[utf8]{inputenc}
# \usepackage[T1]{fontenc}
# \usepackage[english]{babel}
# \usepackage{xcolor}
# \usepackage{hyperref}

# \title{FIREPLACE HAWC+ Polarization Structure Function Report}
# \author{Automated Analysis Pipeline}
# \date{\today}

# \begin{document}
# \maketitle
# """ + METHODOLOGY_TEXT + r"""
# \newpage
# """

# # =========================================================
# # DISCOVER REGION FOLDERS
# # =========================================================
# region_dirs = sorted([p for p in PLOTS_DIR.iterdir() if p.is_dir()])

# if len(region_dirs) == 0:
#     print(f"[WARNING] No region folders found in {PLOTS_DIR}")

# # =========================================================
# # BUILD REPORT PER REGION
# # =========================================================
# for region_dir in region_dirs:
#     object_name = region_dir.name
#     safe_object_name = latex_escape(object_name)

#     stats_file = region_dir / f"{object_name}_stats.json"

#     table_rows = ""

#     if stats_file.exists():
#         with open(stats_file, "r", encoding="utf-8") as f:
#             stats = json.load(f)

#         important_keys = [
#             "file",
#             "object_name",
#             "distance_pc",
#             "pixel_scale_pc",
#             "R_range_pc",
#             "bright_percentile",
#             "I_bright_threshold",
#             "N_good",
#             "N_mask_A_reliable_snr2",
#             "N_mask_B_strict_snr3",
#             "N_mask_C_brightI_snr2",
#             "method",
#             "PI_debiasing",
#         ]

#         for k in important_keys:
#             if k in stats:
#                 table_rows += (
#                     f"        {latex_escape(k)} & {format_value(stats[k])} \\\\\n"
#                 )

#         # Add any remaining keys
#         for k, v in stats.items():
#             if k not in important_keys:
#                 table_rows += (
#                     f"        {latex_escape(k)} & {format_value(v)} \\\\\n"
#                 )
#     else:
#         table_rows += (
#             r"        \textit{Data Status} & \color{red}\textit{Stats JSON missing} \\"
#             + "\n"
#         )

#     # Figure paths relative to REPORT_DIR
#     rel_base = Path("../plots") / object_name

#     f_mask = rel_base / f"{object_name}_mask_comparison.pdf"
#     f_pol = rel_base / f"{object_name}_pol_map_maskA.pdf"
#     f_snr = rel_base / f"{object_name}_snr_colored_maskA.pdf"
#     f_sf_comp = rel_base / f"{object_name}_sf_mask_comparison.pdf"

#     f_sf_A = rel_base / f"{object_name}_sf_A_reliable_snr2.pdf"
#     f_sf_B = rel_base / f"{object_name}_sf_B_strict_snr3.pdf"
#     f_sf_C = rel_base / f"{object_name}_sf_C_brightI_snr2.pdf"

#     # Section
#     latex_content += rf"""
# \section*{{Region: {safe_object_name}}}

# \begin{{table}}[H]
#     \centering
#     \caption*{{Region Parameters and Mask Statistics}}
#     \begin{{tabular}}{{lp{{10cm}}}}
#         \toprule
#         \textbf{{Parameter}} & \textbf{{Value}} \\
#         \midrule
# {table_rows}        \bottomrule
#     \end{{tabular}}
# \end{{table}}
# """

#     # Mask + map figures
#     fig_blocks_1 = ""

#     if (REPORT_DIR / f_mask).exists():
#         fig_blocks_1 += rf"""
#     \begin{{subfigure}}{{0.95\textwidth}}
#         \centering
#         \includegraphics[width=\linewidth]{{{f_mask.as_posix()}}}
#         \caption{{Comparison of the three masks used for the structure-function analysis.}}
#     \end{{subfigure}}
# """

#     if (REPORT_DIR / f_pol).exists():
#         fig_blocks_1 += rf"""
#     \begin{{subfigure}}{{0.48\textwidth}}
#         \centering
#         \includegraphics[width=\linewidth]{{{f_pol.as_posix()}}}
#         \caption{{Magnetic-field pseudo-vectors for Mask A.}}
#     \end{{subfigure}}
# """

#     if (REPORT_DIR / f_snr).exists():
#         fig_blocks_1 += rf"""
#     \begin{{subfigure}}{{0.48\textwidth}}
#         \centering
#         \includegraphics[width=\linewidth]{{{f_snr.as_posix()}}}
#         \caption{{SNR-coded magnetic-field pseudo-vectors for Mask A.}}
#     \end{{subfigure}}
# """

#     if fig_blocks_1.strip():
#         latex_content += rf"""
# \begin{{figure}}[H]
#     \centering
# {fig_blocks_1}
# \end{{figure}}
# """

#     # Structure function comparison
#     if (REPORT_DIR / f_sf_comp).exists():
#         latex_content += rf"""
# \begin{{figure}}[H]
#     \centering
#     \includegraphics[width=0.8\textwidth]{{{f_sf_comp.as_posix()}}}
#     \caption{{Comparison of polarization angle structure functions for the three masks.}}
# \end{{figure}}
# """

#     # Individual SFs
#     fig_blocks_2 = ""

#     if (REPORT_DIR / f_sf_A).exists():
#         fig_blocks_2 += rf"""
#     \begin{{subfigure}}{{0.32\textwidth}}
#         \centering
#         \includegraphics[width=\linewidth]{{{f_sf_A.as_posix()}}}
#         \caption{{Mask A: reliable region.}}
#     \end{{subfigure}}
# """

#     if (REPORT_DIR / f_sf_B).exists():
#         fig_blocks_2 += rf"""
#     \begin{{subfigure}}{{0.32\textwidth}}
#         \centering
#         \includegraphics[width=\linewidth]{{{f_sf_B.as_posix()}}}
#         \caption{{Mask B: strict polarization.}}
#     \end{{subfigure}}
# """

#     if (REPORT_DIR / f_sf_C).exists():
#         fig_blocks_2 += rf"""
#     \begin{{subfigure}}{{0.32\textwidth}}
#         \centering
#         \includegraphics[width=\linewidth]{{{f_sf_C.as_posix()}}}
#         \caption{{Mask C: bright-emission subset.}}
#     \end{{subfigure}}
# """

#     if fig_blocks_2.strip():
#         latex_content += rf"""
# \begin{{figure}}[H]
#     \centering
# {fig_blocks_2}
# \end{{figure}}
# """

#     latex_content += r"\newpage" + "\n"

# # =========================================================
# # END DOCUMENT
# # =========================================================
# latex_content += r"\end{document}"

# tex_path = REPORT_DIR / "assembled_fireplace_report.tex"

# with open(tex_path, "w", encoding="utf-8") as f:
#     f.write(latex_content)

# print(f"[OK] LaTeX report written to: {tex_path}")

In [48]:
from pathlib import Path
import json

# =========================================================
# CONFIG
# =========================================================
PROJECT_NAME = "FIELDMAPS"

OUTPUT_DIR = Path("report/fieldmaps_results")
PLOTS_DIR = OUTPUT_DIR / "plots"
REPORT_DIR = OUTPUT_DIR / "report_text"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_TITLE = f"{PROJECT_NAME} HAWC+ Polarization Structure Function Report"
TEX_FILENAME = "assembled_fieldmaps_report.tex"

# =========================================================
# HELPERS
# =========================================================
def latex_escape(s):
    """
    Escape common LaTeX special characters.
    """
    if s is None:
        return ""
    s = str(s)
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    return "".join(replacements.get(ch, ch) for ch in s)


def format_value(v):
    """
    Format values for LaTeX table.
    """
    if isinstance(v, float):
        return f"{v:.5g}"
    if isinstance(v, int):
        return str(v)
    if isinstance(v, (list, tuple)):
        if all(isinstance(x, (int, float)) for x in v):
            return " -- ".join(f"{x:.5g}" for x in v)
        return latex_escape(v)
    if isinstance(v, dict):
        return latex_escape(json.dumps(v))
    return latex_escape(v)


def include_if_exists(path_from_tex, caption, width=r"\linewidth"):
    """
    Returns a LaTeX subfigure block if the target figure exists.
    path_from_tex is relative to REPORT_DIR.
    """
    full_path = REPORT_DIR / path_from_tex
    if not full_path.exists():
        return ""

    return rf"""
    \begin{{subfigure}}{{0.48\textwidth}}
        \centering
        \includegraphics[width={width}]{{{path_from_tex.as_posix()}}}
        \caption{{{caption}}}
    \end{{subfigure}}
"""


# =========================================================
# METHODOLOGY TEXT
# =========================================================
METHODOLOGY_TEXT = r"""
\section*{Methodology and Data Selection}

This report summarizes the polarization angle structure-function analysis for FIELDMAPS HAWC+ data products.
The same structure-function method as in the FIREPLACE analysis is used here, but applied to the FIELDMAPS target fields.

\paragraph{Input data.}
For each target field, we use maps containing Stokes \(I\), \(Q\), and \(U\), together with the available uncertainty maps and validity masks when present.
The analysis is performed on the science-ready data products prepared for the FIELDMAPS release.

\paragraph{Polarization and debiasing.}
The polarized intensity is computed as
\[
PI = \sqrt{Q^2 + U^2}.
\]
When uncertainty maps are available, the polarized-intensity uncertainty is propagated from the Stokes \(Q\) and \(U\) errors.
We apply the Wardle--Kronberg debiasing correction,
\[
PI_{\rm db} = \sqrt{PI^2 - \sigma_{PI}^2},
\]
with negative values inside the square root clipped to zero.

\paragraph{Masking strategy.}
For each field, the structure function is computed using three masks:
\begin{enumerate}
    \item \textbf{Mask A}: all reliable polarization detections, using \(SNR_{PI} \geq 2\);
    \item \textbf{Mask B}: a stricter polarization-quality subset, using \(SNR_{PI} \geq 3\);
    \item \textbf{Mask C}: a bright-emission subset, using \(SNR_{PI} \geq 2\) and retaining only pixels above a selected percentile of the Stokes \(I\) distribution.
\end{enumerate}
This three-mask approach is used to test how sensitive the structure function is to the adopted polarization-quality threshold and intensity selection.

\paragraph{Structure function.}
For each mask, we compute the polarization angle structure function \(D_\phi(R)\) as a function of projected separation \(R\).
The separation is converted from pixels to physical units using the WCS pixel scale and the adopted source distance.
"""

# =========================================================
# LATEX HEADER
# =========================================================
latex_content = rf"""\documentclass[10pt, a4paper]{{article}}
\usepackage[margin=1.5cm]{{geometry}}
\usepackage{{graphicx}}
\usepackage{{booktabs}}
\usepackage{{subcaption}}
\usepackage{{float}}
\usepackage{{amsmath}}
\usepackage{{mathpazo}}
\usepackage[utf8]{{inputenc}}
\usepackage[T1]{{fontenc}}
\usepackage[english]{{babel}}
\usepackage{{xcolor}}
\usepackage{{hyperref}}

\title{{{REPORT_TITLE}}}
\author{{Automated Analysis Pipeline}}
\date{{\today}}

\begin{{document}}
\maketitle
""" + METHODOLOGY_TEXT + r"""
\newpage
"""

# =========================================================
# DISCOVER FIELD FOLDERS
# =========================================================
field_dirs = sorted([p for p in PLOTS_DIR.iterdir() if p.is_dir()])

if len(field_dirs) == 0:
    print(f"[WARNING] No field folders found in {PLOTS_DIR}")

# =========================================================
# BUILD REPORT PER FIELD
# =========================================================
for field_dir in field_dirs:
    object_name = field_dir.name
    safe_object_name = latex_escape(object_name)

    stats_file = field_dir / f"{object_name}_stats.json"

    table_rows = ""

    if stats_file.exists():
        with open(stats_file, "r", encoding="utf-8") as f:
            stats = json.load(f)

        important_keys = [
            "file",
            "object_name",
            "distance_pc",
            "pixel_scale_pc",
            "R_range_pc",
            "bright_percentile",
            "I_bright_threshold",
            "N_good",
            "N_mask_A_reliable_snr2",
            "N_mask_B_strict_snr3",
            "N_mask_C_brightI_snr2",
            "method",
            "PI_debiasing",
        ]

        for k in important_keys:
            if k in stats:
                table_rows += f"        {latex_escape(k)} & {format_value(stats[k])} \\\\\n"

        for k, v in stats.items():
            if k not in important_keys:
                table_rows += f"        {latex_escape(k)} & {format_value(v)} \\\\\n"

    else:
        table_rows += (
            r"        \textit{Data Status} & \color{red}\textit{Stats JSON missing} \\"
            + "\n"
        )

    # Figure paths relative to REPORT_DIR
    rel_base = Path("../plots") / object_name

    f_mask = rel_base / f"{object_name}_mask_comparison.pdf"
    f_pol = rel_base / f"{object_name}_pol_map_maskA.pdf"
    f_snr = rel_base / f"{object_name}_snr_colored_maskA.pdf"
    f_sf_comp = rel_base / f"{object_name}_sf_mask_comparison.pdf"

    f_sf_A = rel_base / f"{object_name}_sf_A_reliable_snr2.pdf"
    f_sf_B = rel_base / f"{object_name}_sf_B_strict_snr3.pdf"
    f_sf_C = rel_base / f"{object_name}_sf_C_brightI_snr2.pdf"

    # =====================================================
    # SECTION
    # =====================================================
    latex_content += rf"""
\section*{{Field: {safe_object_name}}}

\begin{{table}}[H]
    \centering
    \caption*{{Field Parameters and Mask Statistics}}
    \begin{{tabular}}{{lp{{10cm}}}}
        \toprule
        \textbf{{Parameter}} & \textbf{{Value}} \\
        \midrule
{table_rows}        \bottomrule
    \end{{tabular}}
\end{{table}}
"""

    # =====================================================
    # MASK + MAP FIGURES
    # =====================================================
    fig_blocks_1 = ""

    if (REPORT_DIR / f_mask).exists():
        fig_blocks_1 += rf"""
    \begin{{subfigure}}{{0.95\textwidth}}
        \centering
        \includegraphics[width=\linewidth]{{{f_mask.as_posix()}}}
        \caption{{Comparison of the three masks used for the structure-function analysis.}}
    \end{{subfigure}}
"""

    if (REPORT_DIR / f_pol).exists():
        fig_blocks_1 += rf"""
    \begin{{subfigure}}{{0.48\textwidth}}
        \centering
        \includegraphics[width=\linewidth]{{{f_pol.as_posix()}}}
        \caption{{Magnetic-field pseudo-vectors for Mask A.}}
    \end{{subfigure}}
"""

    if (REPORT_DIR / f_snr).exists():
        fig_blocks_1 += rf"""
    \begin{{subfigure}}{{0.48\textwidth}}
        \centering
        \includegraphics[width=\linewidth]{{{f_snr.as_posix()}}}
        \caption{{SNR-coded magnetic-field pseudo-vectors for Mask A.}}
    \end{{subfigure}}
"""

    if fig_blocks_1.strip():
        latex_content += rf"""
\begin{{figure}}[H]
    \centering
{fig_blocks_1}
\end{{figure}}
"""

    # =====================================================
    # STRUCTURE-FUNCTION COMPARISON
    # =====================================================
    if (REPORT_DIR / f_sf_comp).exists():
        latex_content += rf"""
\begin{{figure}}[H]
    \centering
    \includegraphics[width=0.8\textwidth]{{{f_sf_comp.as_posix()}}}
    \caption{{Comparison of polarization angle structure functions for the three masks.}}
\end{{figure}}
"""

    # =====================================================
    # INDIVIDUAL SF FIGURES
    # =====================================================
    fig_blocks_2 = ""

    if (REPORT_DIR / f_sf_A).exists():
        fig_blocks_2 += rf"""
    \begin{{subfigure}}{{0.32\textwidth}}
        \centering
        \includegraphics[width=\linewidth]{{{f_sf_A.as_posix()}}}
        \caption{{Mask A: reliable polarization detections.}}
    \end{{subfigure}}
"""

    if (REPORT_DIR / f_sf_B).exists():
        fig_blocks_2 += rf"""
    \begin{{subfigure}}{{0.32\textwidth}}
        \centering
        \includegraphics[width=\linewidth]{{{f_sf_B.as_posix()}}}
        \caption{{Mask B: stricter polarization-quality selection.}}
    \end{{subfigure}}
"""

    if (REPORT_DIR / f_sf_C).exists():
        fig_blocks_2 += rf"""
    \begin{{subfigure}}{{0.32\textwidth}}
        \centering
        \includegraphics[width=\linewidth]{{{f_sf_C.as_posix()}}}
        \caption{{Mask C: bright-emission subset.}}
    \end{{subfigure}}
"""

    if fig_blocks_2.strip():
        latex_content += rf"""
\begin{{figure}}[H]
    \centering
{fig_blocks_2}
\end{{figure}}
"""

    latex_content += r"\newpage" + "\n"

# =========================================================
# END DOCUMENT
# =========================================================
latex_content += r"\end{document}"

tex_path = REPORT_DIR / TEX_FILENAME

with open(tex_path, "w", encoding="utf-8") as f:
    f.write(latex_content)

print(f"[OK] LaTeX report written to: {tex_path}")

[OK] LaTeX report written to: report\fieldmaps_results\report_text\assembled_fieldmaps_report.tex


In [51]:
import subprocess
from pathlib import Path

REPORT_DIR = Path("report/fieldmaps_results/report_text")
tex_file = REPORT_DIR / "assembled_fieldmaps_report.tex"

cmd = [
    "pdflatex",
    "-interaction=nonstopmode",
    "-halt-on-error",
    tex_file.name
]

for i in range(2):
    result = subprocess.run(
        cmd,
        cwd=REPORT_DIR,
        capture_output=True,
        text=True
    )

    print(result.stdout[-1000:])

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("pdflatex failed")

pdf_file = REPORT_DIR / "assembled_fieldmaps_report.pdf"
print("PDF saved to:", pdf_file)

lored_maskA.pdf>] [40 <../plots/Snake_F3/Snake
_F3_sf_mask_comparison.pdf> <../plots/Snake_F3/Snake_F3_sf_A_reliable_snr2.pdf>
 <../plots/Snake_F3/Snake_F3_sf_B_strict_snr3.pdf> <../plots/Snake_F3/Snake_F3_
sf_C_brightI_snr2.pdf>] (assembled_fieldmaps_report.aux) )<C:/Users/jeton/AppDa
ta/Local/Programs/MiKTeX/fonts/type1/public/amsfonts/cm/cmex10.pfb><C:/Users/je
ton/AppData/Local/Programs/MiKTeX/fonts/type1/public/amsfonts/cm/cmr10.pfb><C:/
Users/jeton/AppData/Local/Programs/MiKTeX/fonts/type1/public/amsfonts/cm/cmsy10
.pfb><C:/Users/jeton/AppData/Local/Programs/MiKTeX/fonts/type1/public/mathpazo/
fplmri.pfb><C:/Users/jeton/AppData/Local/Programs/MiKTeX/fonts/type1/urw/palati
no/uplb8a.pfb><C:/Users/jeton/AppData/Local/Programs/MiKTeX/fonts/type1/urw/pal
atino/uplr8a.pfb><C:/Users/jeton/AppData/Local/Programs/MiKTeX/fonts/type1/urw/
palatino/uplri8a.pfb>
Output written on assembled_fieldmaps_report.pdf (40 pages, 12652318 bytes).
Transcript written on assembled_fieldmaps_report.log.
